In [ ]:
import sys, subprocess
pkgs = ["segmentation-models-pytorch", "albumentations", "pycocotools", "ultralytics"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ok")


In [ ]:
from pathlib import Path
import json, random, shutil, math
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils
from ultralytics import YOLO

DATA_ROOT = Path("/kaggle/input/datasets/deepxchat/solarfilament/MAGFiLO_1.0_Kaggle_2026")
TRAIN_IMAGES = DATA_ROOT / "train" / "train_images"
TEST_IMAGES = DATA_ROOT / "test" / "test_images"
TRAIN_JSON = DATA_ROOT / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
YOLO_DATA = OUT_DIR / "yolo_detect_data"
# Prefer uploaded yolo11s for this infer run:
YOLO_WEIGHTS = OUT_DIR / "yolo_detect_best.pt"
YOLO_WEIGHTS_FALLBACK = OUT_DIR / "yolo_detect_best.pt"
REFINER_CKPT = Path("/kaggle/input/models/tienti/seed-refiner-magfilo/pytorch/default/1/seed_refiner_best.pt")
SUBMISSION_CSV = OUT_DIR / "submission.csv"
PRED_VAL_CSV = OUT_DIR / "pred_val.csv"
GT_VAL_CSV = OUT_DIR / "gt_val.csv"

# Uncomment ONE:
# RUN_MODE = "dummy"
RUN_MODE = "normal"

# Infer-only: reuse s-detector + refiner
SKIP_YOLO_TRAIN = False
SKIP_REFINER_TRAIN = True

if RUN_MODE == "normal":
    YOLO_MODEL = "yolo11s.pt"
    YOLO_IMGSZ = 1920
    YOLO_EPOCHS = 60
    YOLO_BATCH = 4
    YOLO_CONF = 0.28
    YOLO_IOU = 0.50
    REFINE_CROP = 512
    REFINE_EPOCHS = 12
    REFINE_BATCH = 8
    REFINE_LR = 3e-4
    REFINE_THR = 0.50
    VAL_FRAC = 0.15
    MAX_TRAIN_STEMS = None
    MAX_VAL_STEMS = None
    MAX_TEST_IMAGES = None
elif RUN_MODE == "dummy":
    YOLO_MODEL = "yolov8n.pt"
    YOLO_IMGSZ = 640
    YOLO_EPOCHS = 2
    YOLO_BATCH = 8
    YOLO_CONF = 0.28
    YOLO_IOU = 0.50
    REFINE_CROP = 256
    REFINE_EPOCHS = 2
    REFINE_BATCH = 8
    REFINE_LR = 3e-4
    REFINE_THR = 0.50
    VAL_FRAC = 0.2
    MAX_TRAIN_STEMS = 40
    MAX_VAL_STEMS = 10
    MAX_TEST_IMAGES = 5
else:
    raise ValueError(RUN_MODE)

SEED = 42
WORKERS = 2
PAD_RATIO = 0.40
CONTEXT = 1.5          # square crop side ~ CONTEXT * max(box_w, box_h)
MIN_CROP = 96
MIN_AREA = 200
USE_TTA = False                 # horizontal flip TTA (optional)
USE_BOX_JITTER_TTA = False       # shift seed box, batch forward, mean probs
BOX_JITTER_PX = 8               # ±px on full-res image

if not YOLO_WEIGHTS.exists() and YOLO_WEIGHTS_FALLBACK.exists():
    YOLO_WEIGHTS = YOLO_WEIGHTS_FALLBACK

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count() if DEVICE.type == "cuda" else 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("RUN_MODE:", RUN_MODE)
print("YOLO weights:", YOLO_WEIGHTS, "exists", YOLO_WEIGHTS.exists())
print("REFINER:", REFINER_CKPT, "exists", REFINER_CKPT.exists())
print("YOLO:", YOLO_MODEL, "imgsz", YOLO_IMGSZ, "conf", YOLO_CONF, "iou", YOLO_IOU, "skip_train", SKIP_YOLO_TRAIN)
print("PAD_RATIO", PAD_RATIO, "| skip_refiner", SKIP_REFINER_TRAIN)
print("BOX_JITTER_TTA", USE_BOX_JITTER_TTA, "px", BOX_JITTER_PX, "| flip_TTA", USE_TTA)
print("REFINER crop", REFINE_CROP, "epochs", REFINE_EPOCHS, "batch", REFINE_BATCH)
print("DEVICE:", DEVICE, "N_GPUS:", N_GPUS)
print("DATA:", DATA_ROOT.exists(), TRAIN_JSON.exists())


In [ ]:
print("Loading COCO...")
with open(TRAIN_JSON) as f:
    coco = json.load(f)

images_by_id = {im["id"]: im for im in coco["images"]}
anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

samples = []
missing = 0
for image_id, im in images_by_id.items():
    path = TRAIN_IMAGES / im["file_name"]
    if not path.exists():
        alt = path.with_suffix(".jpg") if path.suffix.lower() == ".jpeg" else path.with_suffix(".jpeg")
        if alt.exists():
            path = alt
        else:
            missing += 1
            continue
    samples.append({
        "image_id": image_id,
        "file_name": im["file_name"],
        "path": path,
        "stem": path.stem,
        "height": im.get("height", 2048),
        "width": im.get("width", 2048),
        "annotations": anns_by_image[image_id],
    })

stems = sorted({s["stem"] for s in samples})
random.shuffle(stems)
n_val = max(1, int(len(stems) * VAL_FRAC))
val_stem_list = stems[:n_val]
train_stem_list = stems[n_val:]
if MAX_TRAIN_STEMS is not None:
    train_stem_list = train_stem_list[:MAX_TRAIN_STEMS]
if MAX_VAL_STEMS is not None:
    val_stem_list = val_stem_list[:MAX_VAL_STEMS]
val_stems, train_stems = set(val_stem_list), set(train_stem_list)
train_samples = [s for s in samples if s["stem"] in train_stems]
val_samples = [s for s in samples if s["stem"] in val_stems]

# per-annotation list for refiner training
def build_ann_list(sample_list):
    rows = []
    for s in sample_list:
        for ann in s["annotations"]:
            if ann.get("iscrowd", 0):
                continue
            if not ann.get("segmentation") and not ann.get("bbox"):
                continue
            rows.append((s, ann))
    return rows

train_ann_list = build_ann_list(train_samples)
val_ann_list = build_ann_list(val_samples)
print(f"samples {len(samples)} missing {missing} | train stems {len(train_stems)} val {len(val_stems)}")
print(f"refiner crops: train {len(train_ann_list)} val {len(val_ann_list)}")


In [ ]:
def build_yolo_detect_split(sample_list, split_name: str):
    img_dir = YOLO_DATA / "images" / split_name
    lbl_dir = YOLO_DATA / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    seen, n_boxes = set(), 0
    for s in sample_list:
        fn = s["file_name"]
        if fn in seen:
            continue
        seen.add(fn)
        dst = img_dir / Path(fn).name
        if not dst.exists():
            shutil.copy2(s["path"], dst)
        w, h = s["width"], s["height"]
        lines = []
        for ann in s["annotations"]:
            x, y, bw, bh = ann["bbox"]
            lines.append(f"0 {(x+bw/2)/w:.6f} {(y+bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}")
            n_boxes += 1
        (lbl_dir / f"{s['stem']}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
    return len(seen), n_boxes


need_yolo_data = not (SKIP_YOLO_TRAIN and YOLO_WEIGHTS.exists())
if need_yolo_data:
    if YOLO_DATA.exists():
        shutil.rmtree(YOLO_DATA)
    n_tr, b_tr = build_yolo_detect_split(train_samples, "train")
    n_va, b_va = build_yolo_detect_split(val_samples, "val")
    yaml_path = YOLO_DATA / "data.yaml"
    yaml_path.write_text(f"path: {YOLO_DATA}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n")
    print(f"YOLO data train={n_tr}/{b_tr} val={n_va}/{b_va}")
else:
    yaml_path = YOLO_DATA / "data.yaml"
    print("Skip building YOLO data; using existing weights")

if SKIP_YOLO_TRAIN and YOLO_WEIGHTS.exists():
    print("Reuse YOLO weights:", YOLO_WEIGHTS)
else:
    if not need_yolo_data:
        raise FileNotFoundError("YOLO weights missing and SKIP_YOLO_TRAIN=True")
    yolo = YOLO(YOLO_MODEL)
    kw = dict(
        data=str(yaml_path), epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ, batch=YOLO_BATCH,
        workers=WORKERS, seed=SEED, project=str(OUT_DIR / "ultralytics_runs"),
        name=f"seedrefine_detect_{RUN_MODE}", exist_ok=True, pretrained=True,
        patience=max(5, YOLO_EPOCHS // 4),
    )
    kw["device"] = 0 if DEVICE.type == "cuda" else "cpu"
    for _bs in (YOLO_BATCH, 2):
        try:
            kw["batch"] = _bs
            yolo.train(**kw)
            break
        except RuntimeError as _e:
            if "out of memory" not in str(_e).lower() or _bs == 2:
                raise
            print("OOM at batch", _bs, "- retrying with batch 2")
            torch.cuda.empty_cache()
            yolo = YOLO(YOLO_MODEL)
    best = Path(yolo.trainer.best) if getattr(getattr(yolo, "trainer", None), "best", None) else None
    if best is None or not best.exists():
        best = OUT_DIR / "ultralytics_runs" / f"seedrefine_detect_{RUN_MODE}" / "weights" / "best.pt"
    if not Path(best).exists():
        raise FileNotFoundError("YOLO best.pt not found")
    YOLO_WEIGHTS = OUT_DIR / "yolo_detect_best.pt"
    shutil.copy2(best, YOLO_WEIGHTS)
    print("YOLO →", YOLO_WEIGHTS)

det_model = YOLO(str(YOLO_WEIGHTS))
print("Detector ready:", YOLO_WEIGHTS)


In [ ]:
def ann_to_mask(ann, h, w):
    seg = ann.get("segmentation")
    if seg:
        rles = mask_utils.frPyObjects(seg, h, w)
        rle = mask_utils.merge(rles) if isinstance(rles, list) else rles
        m = mask_utils.decode(rle)
        if m.ndim == 3:
            m = np.any(m, axis=2)
        return m.astype(np.uint8)
    x, y, bw, bh = map(int, ann["bbox"])
    m = np.zeros((h, w), np.uint8)
    m[y:y+bh, x:x+bw] = 1
    return m


def square_bounds_from_bbox(bbox, h, w, context=CONTEXT, minimum=MIN_CROP):
    x, y, bw, bh = bbox
    cx, cy = x + bw / 2, y + bh / 2
    side = int(math.ceil(max(bw, bh) * context))
    side = max(side, minimum)
    x0 = int(round(cx - side / 2)); y0 = int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0:
        x1 -= x0; x0 = 0
    if y0 < 0:
        y1 -= y0; y0 = 0
    if x1 > w:
        x0 -= x1 - w; x1 = w
    if y1 > h:
        y0 -= y1 - h; y1 = h
    x0, y0 = max(0, x0), max(0, y0)
    return x0, y0, x1, y1


def seed_from_box(h, w, bbox, pad_ratio=PAD_RATIO):
    """Filled padded box seed — same form at train and infer."""
    x, y, bw, bh = bbox
    px, py = bw * pad_ratio, bh * pad_ratio
    x0, y0 = max(0, int(x - px)), max(0, int(y - py))
    x1, y1 = min(w, int(x + bw + px)), min(h, int(y + bh + py))
    seed = np.zeros((h, w), np.uint8)
    seed[y0:y1, x0:x1] = 1
    return seed


def jitter_bbox(bbox, h, w, rng: np.random.Generator):
    """Perturb GT box so train seed looks like noisy YOLO boxes."""
    x, y, bw, bh = map(float, bbox)
    # shift up to ~15% of box size
    x += float(rng.uniform(-0.15, 0.15) * bw)
    y += float(rng.uniform(-0.15, 0.15) * bh)
    # scale side lengths
    sx = float(rng.uniform(0.85, 1.25))
    sy = float(rng.uniform(0.85, 1.25))
    cx, cy = x + bw / 2, y + bh / 2
    bw, bh = max(8.0, bw * sx), max(8.0, bh * sy)
    x, y = cx - bw / 2, cy - bh / 2
    # clamp into image
    x = float(np.clip(x, 0, w - 2))
    y = float(np.clip(y, 0, h - 2))
    bw = float(min(bw, w - x))
    bh = float(min(bh, h - y))
    pad = float(rng.uniform(PAD_RATIO * 0.5, PAD_RATIO * 1.5))
    return [x, y, bw, bh], pad


class SeedRefineDataset(Dataset):
    def __init__(self, ann_list, crop_size, augment: bool):
        self.ann_list = ann_list
        self.crop_size = crop_size
        self.augment = augment
        self._cache = {}

    def __len__(self):
        return len(self.ann_list)

    def _load(self, path):
        key = str(path)
        if key not in self._cache:
            img = cv2.imread(key, cv2.IMREAD_GRAYSCALE)
            self._cache[key] = img
        return self._cache[key]

    def __getitem__(self, idx):
        s, ann = self.ann_list[idx]
        img = self._load(s["path"])
        h, w = img.shape
        target = ann_to_mask(ann, h, w)
        gt_bbox = ann["bbox"]
        # Seed MUST match infer: filled padded box (not GT mask — that leaked the answer).
        if self.augment:
            rng = np.random.default_rng(SEED + idx + random.randint(0, 10_000))
            box_j, pad_j = jitter_bbox(gt_bbox, h, w, rng)
            seed_full = seed_from_box(h, w, box_j, pad_ratio=pad_j)
        else:
            seed_full = seed_from_box(h, w, gt_bbox, pad_ratio=PAD_RATIO)
        x0, y0, x1, y1 = square_bounds_from_bbox(gt_bbox, h, w)
        img_c = cv2.resize(img[y0:y1, x0:x1], (self.crop_size, self.crop_size), interpolation=cv2.INTER_LINEAR)
        tgt_c = cv2.resize(target[y0:y1, x0:x1], (self.crop_size, self.crop_size), interpolation=cv2.INTER_NEAREST)
        seed_c = cv2.resize(seed_full[y0:y1, x0:x1], (self.crop_size, self.crop_size), interpolation=cv2.INTER_NEAREST)
        clahe = cv2.createCLAHE(2.0, (8, 8)).apply(img_c)
        if self.augment and random.random() < 0.5:
            img_c = np.fliplr(img_c).copy(); clahe = np.fliplr(clahe).copy()
            tgt_c = np.fliplr(tgt_c).copy(); seed_c = np.fliplr(seed_c).copy()
        if self.augment and random.random() < 0.5:
            img_c = np.flipud(img_c).copy(); clahe = np.flipud(clahe).copy()
            tgt_c = np.flipud(tgt_c).copy(); seed_c = np.flipud(seed_c).copy()
        x = np.stack([img_c, clahe, seed_c * 255], axis=0).astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]
        x = (x - mean) / std
        y = torch.from_numpy(tgt_c.astype(np.float32))[None]
        return torch.from_numpy(x), y


train_ds = SeedRefineDataset(train_ann_list, REFINE_CROP, True)
val_ds = SeedRefineDataset(val_ann_list, REFINE_CROP, False)
train_loader = DataLoader(train_ds, batch_size=REFINE_BATCH, shuffle=True, num_workers=WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=REFINE_BATCH * 2, shuffle=False, num_workers=WORKERS, pin_memory=True)
xb, yb = next(iter(train_loader))
print("batch", xb.shape, yb.shape, "mask mean", float(yb.mean()))


In [ ]:
def soft_dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(2, 3))
    den = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    return 1.0 - ((2 * inter + eps) / (den + eps)).mean()


@torch.no_grad()
def batch_dice(logits, targets, thr=0.5):
    preds = (torch.sigmoid(logits) > thr).float()
    inter = (preds * targets).sum(dim=(2, 3))
    den = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    return ((2 * inter + 1e-6) / (den + 1e-6)).mean().item()


refiner = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(DEVICE)
if N_GPUS > 1:
    refiner = nn.DataParallel(refiner)

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

optimizer = torch.optim.AdamW(refiner.parameters(), lr=REFINE_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=REFINE_EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

if SKIP_REFINER_TRAIN and REFINER_CKPT.exists():
    state = torch.load(REFINER_CKPT, map_location=DEVICE, weights_only=False)
    unwrap(refiner).load_state_dict(state["model"] if isinstance(state, dict) and "model" in state else state)
    print("Loaded refiner", REFINER_CKPT, "dice", state.get("val_dice") if isinstance(state, dict) else None)
else:
    best_dice = -1.0
    for epoch in range(1, REFINE_EPOCHS + 1):
        refiner.train()
        tr_losses, tr_dices = [], []
        for x, y in tqdm(train_loader, leave=False, desc=f"refine train {epoch}"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = refiner(x)
                loss = 0.45 * F.binary_cross_entropy_with_logits(logits, y) + 0.55 * soft_dice_loss(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            tr_losses.append(loss.item())
            tr_dices.append(batch_dice(logits.detach(), y))
        refiner.eval()
        va_losses, va_dices = [], []
        with torch.no_grad():
            for x, y in tqdm(val_loader, leave=False, desc=f"refine val {epoch}"):
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = refiner(x)
                loss = 0.45 * F.binary_cross_entropy_with_logits(logits, y) + 0.55 * soft_dice_loss(logits, y)
                va_losses.append(loss.item())
                va_dices.append(batch_dice(logits, y))
        scheduler.step()
        tr_l, tr_d = float(np.mean(tr_losses)), float(np.mean(tr_dices))
        va_l, va_d = float(np.mean(va_losses)), float(np.mean(va_dices))
        print(f"Epoch {epoch:02d}/{REFINE_EPOCHS}  train loss={tr_l:.4f} dice={tr_d:.4f} | val loss={va_l:.4f} dice={va_d:.4f}")
        if va_d > best_dice:
            best_dice = va_d
            torch.save({"model": unwrap(refiner).state_dict(), "epoch": epoch, "val_dice": va_d}, REFINER_CKPT)
            print("  saved", REFINER_CKPT, f"val_dice={va_d:.4f}")
    print("Best refiner val dice:", best_dice)
    state = torch.load(REFINER_CKPT, map_location=DEVICE, weights_only=False)
    unwrap(refiner).load_state_dict(state["model"])

refiner.eval()
print("Refiner ready")


In [ ]:
def mask_to_rle(binary_mask: np.ndarray) -> str:
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return rle["counts"].decode("utf-8")


def resolve_overlapping_masks(scored):
    if not scored:
        return []
    scored = sorted(scored, key=lambda t: -t[0])
    occupied = np.zeros_like(scored[0][1], dtype=bool)
    out = []
    for score, m in scored:
        m = m.copy(); m[occupied] = 0
        if int(m.sum()) == 0:
            continue
        occupied |= m.astype(bool)
        out.append(m.astype(np.uint8))
    return out


@torch.no_grad()
def refine_box_on_image(gray: np.ndarray, bbox_xyxy, conf: float) -> tuple[np.ndarray, float]:
    """Refine one YOLO box. Optional box-jitter TTA: same crop, shifted seeds, one batched forward."""
    h, w = gray.shape
    x1, y1, x2, y2 = bbox_xyxy
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    bbox = [x1, y1, bw, bh]
    x0, y0, x1b, y1b = square_bounds_from_bbox(bbox, h, w)
    img_c = cv2.resize(gray[y0:y1b, x0:x1b], (REFINE_CROP, REFINE_CROP), interpolation=cv2.INTER_LINEAR)
    clahe = cv2.createCLAHE(2.0, (8, 8)).apply(img_c)
    mean = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]
    std = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]

    if USE_BOX_JITTER_TTA:
        j = int(BOX_JITTER_PX)
        shifts = [(0, 0), (j, 0), (-j, 0), (0, j), (0, -j)]
    else:
        shifts = [(0, 0)]

    batch = []
    for dx, dy in shifts:
        bbox_j = [x1 + dx, y1 + dy, bw, bh]
        seed_full = seed_from_box(h, w, bbox_j)
        seed_c = cv2.resize(seed_full[y0:y1b, x0:x1b], (REFINE_CROP, REFINE_CROP), interpolation=cv2.INTER_NEAREST)
        x = np.stack([img_c, clahe, seed_c * 255], 0).astype(np.float32) / 255.0
        batch.append((x - mean) / std)
    x = torch.from_numpy(np.stack(batch, 0)).to(DEVICE)  # (B, 3, H, W)
    logits = refiner(x)
    if USE_TTA:
        logits = 0.5 * (logits + refiner(x.flip(-1)).flip(-1))
    prob = torch.sigmoid(logits).mean(dim=0)[0].cpu().numpy()
    crop_m = (prob >= REFINE_THR).astype(np.uint8)
    full = np.zeros((h, w), np.uint8)
    full[y0:y1b, x0:x1b] = cv2.resize(crop_m, (x1b - x0, y1b - y0), interpolation=cv2.INTER_NEAREST)
    return full, float(conf)


def seedrefine_predict(image_path: Path) -> list[np.ndarray]:
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(image_path)
    if img.shape != (2048, 2048):
        img = cv2.resize(img, (2048, 2048), interpolation=cv2.INTER_AREA)
    bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    res = det_model.predict(
        source=bgr, conf=YOLO_CONF, iou=YOLO_IOU, imgsz=YOLO_IMGSZ,
        verbose=False, device=0 if DEVICE.type == "cuda" else "cpu",
    )[0]
    scored = []
    if res.boxes is not None and len(res.boxes):
        xyxy = res.boxes.xyxy.detach().cpu().numpy()
        confs = res.boxes.conf.detach().cpu().numpy()
        for i in np.argsort(-confs):
            m, sc = refine_box_on_image(img, xyxy[i], float(confs[i]))
            if int(m.sum()) >= MIN_AREA:
                scored.append((sc, m))
    return resolve_overlapping_masks(scored)


def predict_image_to_rows(image_path: Path, image_id: str | None = None) -> list[dict]:
    image_id = image_id or image_path.stem
    masks = seedrefine_predict(image_path)
    return [{"filament_id": f"{image_id}_{i}", "segmentation_rle": mask_to_rle(m)} for i, m in enumerate(masks, start=1)]


val_paths_by_stem = {s["stem"]: s["path"] for s in val_samples}
if val_paths_by_stem:
    demo = next(iter(val_paths_by_stem.values()))
    print(f"Demo {demo.name}: {len(predict_image_to_rows(demo))} instances")


In [ ]:

# ---- Held-out PQ grid: new high-res detector vs public baseline detector ----
import time
BASE_YOLO = Path("/kaggle/input/models/tienti/yolo-magfilo/pytorch/default/1/yolo_detect_best.pt")
CONF_FLOOR = 0.10
CONF_GRID = [0.10, 0.15, 0.20, 0.28, 0.35, 0.45, 0.55]
POST_AREA_GRID = [0, 200]
_dev = 0 if DEVICE.type == "cuda" else "cpu"

def _read_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    if img.shape != (2048, 2048):
        img = cv2.resize(img, (2048, 2048), interpolation=cv2.INTER_AREA)
    return img

@torch.no_grad()
def detect_refine(det, path, imgsz):
    """All refined instances down to CONF_FLOOR as (conf, rle), best first."""
    img = _read_gray(path)
    res = det.predict(source=cv2.cvtColor(img, cv2.COLOR_GRAY2BGR), conf=CONF_FLOOR, iou=YOLO_IOU,
                      imgsz=imgsz, verbose=False, device=_dev)[0]
    out = []
    if res.boxes is not None and len(res.boxes):
        xyxy = res.boxes.xyxy.detach().cpu().numpy()
        confs = res.boxes.conf.detach().cpu().numpy()
        for i in np.argsort(-confs):
            m, sc = refine_box_on_image(img, xyxy[i], float(confs[i]))
            if int(m.sum()) >= MIN_AREA:
                out.append((sc, mask_utils.encode(np.asfortranarray(m))))
    return out

def assemble(cands, conf, post_area):
    """Disjoint instances for one image at a given confidence, as RLE dicts."""
    occupied = np.zeros((2048, 2048), bool)
    out = []
    for sc, rle in cands:
        if sc < conf:
            break
        m = mask_utils.decode(rle).astype(bool) & ~occupied
        if int(m.sum()) < max(post_area, 1):
            continue
        occupied |= m
        out.append(mask_utils.encode(np.asfortranarray(m.astype(np.uint8))))
    return out

def official_pq(gt_groups, pred_by_stem):
    tp = fp = fn = 0; s = 0.0
    for stem, g in gt_groups:
        p = pred_by_stem.get(stem, [])
        if not p:
            fn += len(g); continue
        I = np.asarray(mask_utils.iou(g, p, [0] * len(p)))
        Hh = I > 0.5
        tp += int(Hh.sum()); s += float(I[Hh].sum())
        fp += int((Hh.sum(0) == 0).sum()); fn += int((Hh.sum(1) == 0).sum())
    d = tp + 0.5 * fp + 0.5 * fn
    return {"pq": round(s / d, 5) if d else 0.0, "tp": tp, "fp": fp, "fn": fn,
            "mean_tp_iou": round(s / tp, 4) if tp else 0.0}

def _ann_rle(ann, h, w):
    rles = mask_utils.frPyObjects(ann["segmentation"], h, w)
    return mask_utils.merge(rles) if isinstance(rles, list) else rles

# one GT group per annotator-image, exactly like the organizer's evaluator
gt_groups, gt_rows = [], []
for s_ in val_samples:
    g = []
    for k, ann in enumerate(s_["annotations"], start=1):
        if not ann.get("segmentation"):
            continue
        r = _ann_rle(ann, s_["height"], s_["width"])
        g.append(r)
        c = r["counts"]
        gt_rows.append({"filament_id": f"{s_['image_id']}_{k}", "segmentation_rle": c.decode("utf-8") if isinstance(c, bytes) else c})
    if g:
        gt_groups.append((s_["stem"], g))
pd.DataFrame(gt_rows, columns=["filament_id", "segmentation_rle"]).to_csv(GT_VAL_CSV, index=False)
print("gt groups", len(gt_groups), "val images", len(val_paths_by_stem))

detectors = {"new_hires": (det_model, YOLO_IMGSZ)}
if BASE_YOLO.exists():
    detectors["public_baseline"] = (YOLO(str(BASE_YOLO)), 1280)
else:
    print("WARNING: public baseline detector not found:", BASE_YOLO)

report = {"config": {"yolo_imgsz": YOLO_IMGSZ, "yolo_epochs": YOLO_EPOCHS, "refiner": str(REFINER_CKPT),
                     "refine_thr": REFINE_THR, "pad_ratio": PAD_RATIO, "min_area": MIN_AREA}, "grid": []}
val_cands = {}
for name, (det, imgsz) in detectors.items():
    t0 = time.time()
    val_cands[name] = {stem: detect_refine(det, path, imgsz) for stem, path in val_paths_by_stem.items()}
    print(name, "val inference", round(time.time() - t0, 1), "s")
    for conf in CONF_GRID:
        for pa in POST_AREA_GRID:
            pred = {stem: assemble(c, conf, pa) for stem, c in val_cands[name].items()}
            row = {"detector": name, "conf": conf, "post_area": pa, **official_pq(gt_groups, pred),
                   "pred_rows": sum(len(v) for v in pred.values())}
            report["grid"].append(row)
            print("PQGRID", json.dumps(row))

best = {n: max((r for r in report["grid"] if r["detector"] == n), key=lambda r: r["pq"]) for n in detectors}
report["best"] = best
print("BEST", json.dumps(best))

# validation predictions + test submission from the NEW detector at its best setting
b = best["new_hires"]
rows = []
for stem, c in val_cands["new_hires"].items():
    for i, r in enumerate(assemble(c, b["conf"], b["post_area"]), start=1):
        rows.append({"filament_id": f"{stem}_{i}", "segmentation_rle": r["counts"].decode("utf-8")})
pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"]).to_csv(PRED_VAL_CSV, index=False)

test_paths = sorted(TEST_IMAGES.glob("*.jpeg")) + sorted(TEST_IMAGES.glob("*.jpg"))
sub_rows, fallback = [], 0
for path in tqdm(test_paths, desc="test"):
    c = detect_refine(det_model, path, YOLO_IMGSZ)
    inst = assemble(c, b["conf"], b["post_area"])
    if not inst and c:   # keep every observation covered: best low-confidence candidate
        inst = assemble(c[:1], 0.0, 0); fallback += 1
    for i, r in enumerate(inst, start=1):
        sub_rows.append({"filament_id": f"{path.stem}_{i}", "segmentation_rle": r["counts"].decode("utf-8")})
sub_df = pd.DataFrame(sub_rows, columns=["filament_id", "segmentation_rle"])
sub_df.to_csv(SUBMISSION_CSV, index=False)
report["test"] = {"rows": len(sub_df), "images": len(test_paths),
                  "covered": int(sub_df["filament_id"].str.rsplit("_", n=1).str[0].nunique()) if len(sub_df) else 0,
                  "fallback_images": fallback}
(OUT_DIR / "run_report.json").write_text(json.dumps(report, indent=2))
print("TEST", json.dumps(report["test"]))
